In [ ]:
!pip -q install -U pip

# pin lại (khuyến nghị để giảm xung đột với Colab)
!pip -q install --force-reinstall "numpy==2.0.2" "pandas==2.2.2"

# transformers 4.x + deps đồng bộ
!pip -q install --force-reinstall \
  "transformers==4.41.2" \
  "tokenizers==0.19.1" \
  "accelerate==0.30.1" \
  "huggingface-hub==0.23.4" \
  "safetensors>=0.4.3" \
  "bert-score==0.3.13"

In [ ]:
!pip -q install pandas numpy tqdm bert-score
!pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install rouge-score
!pip install --force-reinstall rouge-score

In [ ]:
import os, json, ast, re, time, subprocess
import numpy as np
import pandas as pd
import psutil
from tqdm import tqdm
import torch

from bert_score import score as bert_score
from llama_cpp import Llama
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

torch.set_num_threads(4)       
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA is not available. Please enable GPU runtime in Colab."
print("Imports OK: bert_score + llama_cpp + rouge_scorer + nltk")

BM25_PATH         = "/content/drive/MyDrive/lap/lap_lap/pqaL_test_biobert_bm25_top5.csv"
NEW_LABEL_PATH    = "/content/drive/MyDrive/biobert_large/stage2_large_val_results_full.csv"
MODEL_PATH        = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/llama-2-13b-chat.Q4_0.gguf"
OUT_JSON          = "/content/drive/MyDrive/lap/lap_lap_lap/pqal_eval_500_k3_prednew_13b.json"
OUT_METRICS_CSV   = "/content/drive/MyDrive/lap/lap_lap_lap/pqal_metrics_k3_prednew_13b.csv"
OUT_TABLE_MD      = "/content/drive/MyDrive/lap/lap_lap_lap/pqal_table_k3_prednew_13b.md"

BERTSCORE_MODEL    = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
N_SAMPLES          = 500
SEED               = 42
N_CTX              = 2048
MAX_TOKENS         = 100     
TEMPERATURE        = 0.2
TOP_P              = 0.9
N_GPU_LAYERS       = 35
MAX_DOC_CHARS      = 900
MAX_EVIDENCE_DOCS_CAP = 5
K_TARGET           = 3

np.random.seed(SEED)

def get_memory_usage():
    """Lấy thông tin RAM (psutil) và VRAM GPU (nvidia-smi)"""
    vm = psutil.virtual_memory()
    ram_used_mb = (vm.total - vm.available) / (1024**2)
    gpu_used_mb = 0.0
    try:
        smi_out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        gpu_used_mb = float(smi_out.strip().split('\n')[0])
    except Exception:
        if torch.cuda.is_available():
            gpu_used_mb = torch.cuda.memory_allocated() / (1024**2)
    return ram_used_mb, gpu_used_mb

def load_bm25_file(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t", dtype=str)
        if "pubid" in df.columns and "question" in df.columns:
            return df
    except Exception:
        pass
    return pd.read_csv(path, sep=",", dtype=str)

def parse_list_cell(x):
    if x is None:
        return []
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return []
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return []

def build_evidence_from_topk_row(row: pd.Series, k: int):
    docs = parse_list_cell(row.get(f"top_{k}", ""))
    docs = [str(d).strip() for d in docs if str(d).strip()]
    return docs[:min(k, MAX_EVIDENCE_DOCS_CAP, len(docs))]

def first_sentence(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return text
    parts = re.split(r'(?<=[.!?])\s+', text)
    return parts[0].strip() if parts else text

bm25_df = load_bm25_file(BM25_PATH)
bm25_df["question"] = bm25_df["question"].astype(str).str.strip()
val_df = pd.read_csv(NEW_LABEL_PATH, sep=",", dtype=str)
val_df.columns = [col.strip() for col in val_df.columns]

col_map = {}
for col in val_df.columns:
    if col.lower().strip() == "question":
        col_map[col] = "question"
    elif col.lower().strip() == "long_answer":
        col_map[col] = "long_answer"
    elif col.lower().strip() == "final_decision":
        col_map[col] = "final_decision"
    elif col.lower().strip() == "predicted_label":
        col_map[col] = "predicted_label"
val_df = val_df.rename(columns=col_map)
keep_cols = ['question', 'long_answer', 'final_decision', 'predicted_label']
for col in keep_cols:
    if col not in val_df.columns:
        raise ValueError(f"File mới thiếu cột: {col}")
val_df = val_df[keep_cols]
val_df["question"] = val_df["question"].astype(str).str.strip()

df = bm25_df.merge(val_df, on="question", how="inner")
df = df[(df["question"].str.len() > 10) & (df["long_answer"].str.len() > 20)]
df_sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print(f"Using sample: {df_sample.shape}. Target K = {K_TARGET}")

print("\nLoading Llama (GPU offload)...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=16,        
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)
ram_mb, gpu_mb = get_memory_usage()
print(f"Llama loaded. RAM: {ram_mb:.1f} MB | VRAM: {gpu_mb:.1f} MB\n")

def llama_generate_final(question: str, evidence_docs: list, predicted_label: str) -> str:
    evidence_block = "\n".join([f"{i+1}. {doc[:MAX_DOC_CHARS]}" for i, doc in enumerate(evidence_docs)])
    prompt = f"""[INST] You are a medical QA assistant.
Use ONLY the evidence. If evidence is insufficient, say "Insufficient evidence."

Question: {question}
Predicted answer: {predicted_label}

Evidence:
{evidence_block}

Write ONLY ONE final concluding sentence answering the question (no extra explanation). [/INST]
"""
    out = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        stop=["</s>", "[INST]", "[/INST]"],
        echo=False
    )
    return first_sentence(out["choices"][0]["text"].strip())

results = []
gen_time_sum_ms = 0.0
gen_time_n_ok = 0
t_gen_all0 = time.time()

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    q = row["question"]
    gt_long = row["long_answer"]
    gt_label = row.get("final_decision", "")
    pred_label = row.get("predicted_label", "")
    item = {
        "question": q,
        "ground_truth_long": gt_long,
        "ground_truth_label": gt_label,
        "predicted_label": pred_label,
        "per_k": {}
    }
    evidence_docs = build_evidence_from_topk_row(row, K_TARGET)
    ram_mb, gpu_mb = get_memory_usage()
    try:
        t0 = time.time()
        gen = llama_generate_final(q, evidence_docs, pred_label)
        gen_ms = (time.time() - t0) * 1000.0
        gen_time_sum_ms += gen_ms
        gen_time_n_ok += 1
        item["per_k"][str(K_TARGET)] = {
            "k": K_TARGET,
            "generated": gen,
            "evidence_docs": evidence_docs,
            "gen_ms": float(gen_ms),
            "ram_used_mb": float(ram_mb),
            "gpu_used_mb": float(gpu_mb)
        }
    except Exception as e:
        item["per_k"][str(K_TARGET)] = {
            "k": K_TARGET,
            "generated": "",
            "evidence_docs": evidence_docs,
            "gen_ms": None,
            "error": str(e),
            "ram_used_mb": float(ram_mb),
            "gpu_used_mb": float(gpu_mb)
        }
    results.append(item)

gen_elapsed_s = time.time() - t_gen_all0
print(f"Generation done. items: {len(results)} | elapsed: {gen_elapsed_s:.1f}s")
avg_ms = (gen_time_sum_ms / gen_time_n_ok) if gen_time_n_ok > 0 else float("nan")
throughput = len(results) / gen_elapsed_s if gen_elapsed_s > 0 else float("nan")
print(f"AVG gen time (k={K_TARGET}): {avg_ms:.2f} ms/item, Throughput: {throughput:.3f} samples/sec")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

metrics = []
cands, refs = [], []
rouge1s, rouge2s, rougels, bleus = [], [], [], []
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)  

for item in results:
    gen = item["per_k"].get(str(K_TARGET), {}).get("generated", "")
    ref = item.get("ground_truth_long", "")
    if gen and ref:
        cands.append(gen)
        refs.append(ref)
        scores = scorer.score(ref, gen)
        rouge1s.append(scores['rouge1'].fmeasure)
        rouge2s.append(scores['rouge2'].fmeasure)
        rougels.append(scores['rougeL'].fmeasure)
        ref_tokens = [ref.split()]
        gen_tokens = gen.split()
        bleu = sentence_bleu(ref_tokens, gen_tokens, smoothing_function=SmoothingFunction().method1)
        bleus.append(bleu)

n = len(cands)
ram_b, gpu_b = get_memory_usage()
t0 = time.time()
P, R, F1 = bert_score(
    cands, refs,
    lang="en",
    model_type=BERTSCORE_MODEL,
    num_layers=12,                    
    batch_size=64,                      
    verbose=False,
    rescale_with_baseline=False,       
    device="cuda"
)
bs_ms = (time.time() - t0) * 1000.0
ram_a, gpu_a = get_memory_usage()
P = P.detach().cpu().numpy()
R = R.detach().cpu().numpy()
F1 = F1.detach().cpu().numpy()

metrics.append({
    "k": K_TARGET,
    "n": n,
    "rouge1_mean": float(np.mean(rouge1s)),
    "rouge2_mean": float(np.mean(rouge2s)),
    "rougeL_mean": float(np.mean(rougels)),
    "bleu_mean": float(np.mean(bleus)),
    "bertscore_P_mean": float(P.mean()),
    "bertscore_R_mean": float(R.mean()),
    "bertscore_F1_mean": float(F1.mean()),
    "bertscore_F1_std": float(F1.std()),
    "avg_gen_ms": avg_ms,
    "total_gen_time_s": float(gen_elapsed_s),
    "throughput": float(throughput),
    "bertscore_total_ms": float(bs_ms),
    "bertscore_avg_ms_per_pair": float(bs_ms / max(n, 1)),
    "ram_before_bs_mb": ram_b,
    "gpu_before_bs_mb": gpu_b,
    "gpu_after_bs_mb": gpu_a
})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
display(metrics_df)

md = []
md.append("| k | n | R1 | R2 | RL | BLEU | F1 mean | F1 std | avg_gen_ms | throughput | bs_total_ms | bs_avg_ms/pair | GPU VRAM (MB) |")
md.append("|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for _, r in metrics_df.iterrows():
    k = int(r["k"])
    n = int(r["n"])
    r1 = r["rouge1_mean"]
    r2 = r["rouge2_mean"]
    rl = r["rougeL_mean"]
    bleu = r["bleu_mean"]
    f1m = r["bertscore_F1_mean"]
    f1s = r["bertscore_F1_std"]
    ag  = r["avg_gen_ms"]
    tp  = r["throughput"]
    bt  = r["bertscore_total_ms"]
    bap = r["bertscore_avg_ms_per_pair"]
    gpu = r.get("gpu_after_bs_mb", float('nan'))

    md.append(
        f"| {k} | {n} | "
        f"{r1:.4f} | {r2:.4f} | {rl:.4f} | {bleu:.4f} | "
        f"{f1m:.4f} | {f1s:.4f} | {ag:.2f} | {tp:.2f} | {bt:.2f} | {bap:.2f} | {gpu:.1f} |"
    )

with open(OUT_TABLE_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md))

print(f"\nSaved Markdown Table:\n")
print("\n".join(md))

In [ ]:
!pip install sacrebleu

In [ ]:
import json
import numpy as np
import pandas as pd
from bert_score import score as bert_score
from rouge_score import rouge_scorer
import sacrebleu

OUT_JSON = "/content/drive/MyDrive/lap/lap_lap_lap/pqal_eval_500_k3_prednew_13b.json"
OUT_METRICS_CSV = "/content/drive/MyDrive/lap/lap_lap_lap/pqal_metrics_k3_prednew_13b.csv"

BERTSCORE_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

# Khởi tạo công cụ tính ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

with open(OUT_JSON, "r", encoding="utf-8") as f:
    results = json.load(f)

metrics = []
for k in range(1, 6):
    cands, refs = [], []
    for item in results:
        gen = item["per_k"].get(str(k), {}).get("generated", "")
        ref = item.get("ground_truth_long", "")
        if gen and ref:
            cands.append(gen)
            refs.append(ref)

    n = len(cands)
    if n == 0:
        metrics.append({
            "k": k, "n": 0,
            "bertscore_P_mean": np.nan,
            "bertscore_R_mean": np.nan,
            "bertscore_F1_mean": np.nan,
            "bertscore_F1_std": np.nan,
            "rouge1_f1_mean": np.nan,
            "rouge2_f1_mean": np.nan,
            "rougeL_f1_mean": np.nan,
            "bleu_mean": np.nan
        })
        continue

    # --- 1. Tính BERTScore ---
    P, R, F1_bert = bert_score(
        cands, refs,
        model_type=BERTSCORE_MODEL,
        num_layers=12,               # FIX
        lang="en",
        verbose=False,
        rescale_with_baseline=False, # Khuyến nghị cho model custom
        device="cuda"
    )

    P = P.detach().cpu().numpy()
    R = R.detach().cpu().numpy()
    F1_bert = F1_bert.detach().cpu().numpy()

    # --- 2. Tính ROUGE và BLEU ---
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    bleu_scores = []

    for cand, ref in zip(cands, refs):
        # ROUGE
        r_scores = scorer.score(ref, cand)
        rouge1_scores.append(r_scores['rouge1'].fmeasure)
        rouge2_scores.append(r_scores['rouge2'].fmeasure)
        rougeL_scores.append(r_scores['rougeL'].fmeasure)

        # BLEU (sacrebleu nhận vào 1 chuỗi ứng viên và 1 list các chuỗi tham chiếu)
        # Chia 100 để đưa về scale 0-1 giống với BERTScore và ROUGE
        b_score = sacrebleu.sentence_bleu(cand, [ref]).score / 100.0
        bleu_scores.append(b_score)

    # --- 3. Lưu tổng hợp ---
    metrics.append({
        "k": k,
        "n": n,
        "bertscore_P_mean": float(P.mean()),
        "bertscore_R_mean": float(R.mean()),
        "bertscore_F1_mean": float(F1_bert.mean()),
        "bertscore_F1_std": float(F1_bert.std()),
        "rouge1_f1_mean": float(np.mean(rouge1_scores)),
        "rouge2_f1_mean": float(np.mean(rouge2_scores)),
        "rougeL_f1_mean": float(np.mean(rougeL_scores)),
        "bleu_mean": float(np.mean(bleu_scores)),
    })

metrics_df = pd.DataFrame(metrics).sort_values("k")
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
print("Saved:", OUT_METRICS_CSV)
display(metrics_df)